# Notebook 2 — Multi-Criteria Risk Filtering

**Goal:** move beyond "look up one dam" to "search across all 234 dams with
AND/OR logic," using `POST /risk/metrics/filters`.

## The ordinal scale — and why it exists

Every numeric field in this dataset could be filtered on its raw value, but raw
values aren't how most people think about risk. A dam safety officer doesn't want to
remember that a census tract's SVI percentile of "0.81" is concerning — they want to
ask for dams near a **"Very High"** vulnerability area, the same way they'd ask for
"High Risk" without needing to know what raw cutoff that corresponds to. That's the
whole point of the **ordinal risk level**, a 0–4 scale layered on top of the raw
numbers:

| Ordinal | Meaning |
|---|---|
| 0 | No Risk |
| 1 | Low Risk |
| 2 | Moderate Risk |
| 3 | High Risk |
| 4 | Very High Risk |

This REST API isn't the original consumer of that idea — the
[dam risk dashboard](https://dams.i-guide.io/) is. Open its **Filters** panel (the
funnel icon next to the search box) and you'll find the same concept as a UI: a
"Select Field Name" dropdown grouped by category (`Population | SVI`,
`Hospitals | Beds`, `Major Roads | Interstates (Miles)`, and so on), and five
checkboxes — **No Risk, Low, Moderate, High, Very High** — that you can combine, so
a dam safety officer can ask for "SVI is High *or* Very High" without ever seeing a
raw score. `POST /risk/metrics/filters` is that same idea exposed programmatically —
scripting the same kind of query the dashboard's checkboxes build for you by hand.


In [ ]:
import requests
import pandas as pd

BASE_URL = "http://149.165.154.170:30080"


def filter_dams(logic, filters, n=20):
    """POST /risk/metrics/filters and return the results as a DataFrame."""
    resp = requests.post(
        f"{BASE_URL}/risk/metrics/filters",
        json={"logic": logic, "filters": filters, "n": n},
        timeout=15,
    )
    resp.raise_for_status()
    payload = resp.json()
    print(f"logic={payload['logic']}  filters_count={payload['filters_count']}  results_count={payload['results_count']}")
    return pd.DataFrame(payload["results"])


Not every variable is covered by this ordinal system — `hospital_num_beds` isn't,
for instance (more on that in Challenge 2), and `hazard_potential_classification` /
`state` are plain text fields filtered with `eq`/`in`, not ordinals at all. For
everything else, `GET /risk/metrics/ordinals` lets you look up the exact breakpoints
for any variable, live — it used to return an HTTP 500 error on this server (an
earlier version of this notebook had to fall back to a static reference image
because of it), but that's since been fixed, so let's just call it directly.


In [ ]:
ordinals = requests.get(f"{BASE_URL}/risk/metrics/ordinals", timeout=10).json()
ordinals["variables"]["svi_score"]


That matches the fixed direction we'll rely on below: ordinal 1 ("Low Risk") is the
*lowest* raw SVI range, ordinal 4 ("Very High Risk") is the *highest* — the standard
convention, and what the dashboard has always shown. Here's the project team's own
reference table for every variable at once, useful when you want the whole picture
without querying one variable at a time:

![Ordinal breakpoint reference table](png/ordinal_breaks_revised.png)


## Challenge 1 — Social Vulnerability Index thresholds

Find high-hazard dams (per NID classification) where the surrounding area also has a
**High** SVI ordinal (3) or worse. This is an `AND` of two filters — one text `eq`,
one ordinal `gte`.


In [ ]:
svi_high = filter_dams(
    logic="AND",
    filters=[
        {"variable": "hazard_potential_classification", "op": "eq", "value": "High"},
        {"variable": "svi_score", "op": "gte", "value": 3},
    ],
    n=15,
)
svi_high[["damnumber", "dam_name", "hazard_potential_classification", "svi_score"]]


Look at the `svi_score` column in the table above — every value is 0.5 or higher,
consistent with "High or Very High risk" (ordinal ≥ 3) meaning a *high* raw
percentile. That matches both the reference table above and the dashboard's own
Filters panel, which is exactly what you'd want: don't take that agreement on faith,
though — check it yourself, the same way you'd check any API you're about to build
an analysis on top of. We can do that with the same filter endpoint, just fixing the
ordinal and looking at the spread of raw values it returns for each level.


In [ ]:
for ordinal in range(5):
    bucket = filter_dams(logic="AND", filters=[{"variable": "svi_score", "op": "eq", "value": ordinal}], n=500)
    if bucket.empty:
        print(f"  ordinal={ordinal}  n=0")
        continue
    scores = bucket["svi_score"]
    print(f"  ordinal={ordinal}  n={len(scores):3d}  raw svi_score range=[{scores.min():.4f}, {scores.max():.4f}]")


Confirmed: the ranges climb monotonically with the ordinal — ordinal 1 ("Low Risk")
holds the lowest raw scores (~0.0004–0.25), ordinal 4 ("Very High Risk") holds the
highest (~0.76–0.79) — matching the reference table above and the dashboard's own
Filters panel. (An earlier version of this API had this backwards for `svi_score`
specifically — the same empirical check above is what caught it. Worth remembering:
"the docs say so" and "the dashboard agrees" are both good signs, but a thirty-second
check like this one is cheap insurance against relying on either blindly.)


## Challenge 2 — Hospital beds affected

Naive first attempt: filter on `hospital_num_beds` directly. Watch what happens —
run it with and without the filter and compare the results.


In [ ]:
with_filter = filter_dams(logic="AND", filters=[{"variable": "hospital_num_beds", "op": "gte", "value": 1}], n=5)
no_filter = filter_dams(logic="AND", filters=[], n=5)

print("same dams both times:", with_filter["damnumber"].tolist() == no_filter["damnumber"].tolist())


Identical results — the filter did nothing. This matches what the API docs warn:
*"Variables without ordinal ranges defined (e.g., `hospital_num_beds`) are silently
ignored if included in filters."* `hospital_num_beds` is a raw count, not one of the
ordinal-bucketed variables, so `/risk/metrics/filters` can't use it.

**The fix:** pull the raw metrics table for every dam (`damnumber=all`) and sort it
yourself with pandas — exactly the kind of thing `/risk/metrics` is for.


In [ ]:
all_beds = requests.get(
    f"{BASE_URL}/risk/metrics",
    params={"damnumber": "all", "targets": "hospital_num_beds"},
    timeout=15,
).json()

beds_df = pd.DataFrame(all_beds["items"])
top_beds = beds_df.sort_values("hospital_num_beds", ascending=False).head(10)
top_beds[["damnumber", "dam_name", "county", "hospital_num_beds"]]


## Challenge 3 — Interstate miles affected

Unlike `hospital_num_beds`, `total_interstate_impact_mile` **is** ordinal-filterable
— we can combine an ordinal filter with a state filter, then separately rank the
matches by their actual mile value using `/risk/summary/top`.


In [ ]:
interstate_high = filter_dams(
    logic="AND",
    filters=[
        {"variable": "state", "op": "eq", "value": "UT"},
        {"variable": "total_interstate_impact_mile", "op": "gte", "value": 3},
    ],
    n=15,
)
interstate_high[["damnumber", "dam_name", "total_interstate_impact_mile"]]


In [ ]:
# Rank ALL dams by raw interstate miles affected (not just ordinal-3-and-up):
top_interstate = requests.get(
    f"{BASE_URL}/risk/summary/top", params={"target": "total_interstate_impact_mile", "n": 10}, timeout=10
).json()
pd.DataFrame(top_interstate["top"])


## Exercise

Build your own query using `OR` logic: find dams where *either* `svi_score` is Very
High (4) *or* `total_interstate_impact_mile` is Very High (4). Use the `filter_dams`
helper above — just change `logic` to `"OR"`.


In [ ]:
# your query here
